# N2: Cross-Encoder Reranker Fine-Tuning (BAAI/bge-reranker-v2-m3 on Hard Negatives)

- **Base Model**: `BAAI/bge-reranker-v2-m3`
- **Dataset**: Hard negatives mined from SQLite thumbs-down feedback logs + cross-state confusers
- **Objective**: BCEWithLogitsLoss over (query, positive_scheme, hard_negative_scheme) triples
- **Target**: MRR@5 improvement on multilingual cross-state disambiguation
- **Seed**: `42` (pinned across Python, NumPy, PyTorch)
- **Hardware**: Kaggle / Colab T4 (16 GB VRAM). Set `SAMPLE_FRAC=0.05` for top-to-bottom smoke test.


In [ ]:
# Pinned versions for reproducible Kaggle T4 execution
!pip install -q transformers==4.47.1 sentence-transformers==3.3.1 peft==0.14.0 trl==0.13.0 bitsandbytes==0.45.0 mlflow==2.19.0 huggingface_hub==0.27.0


In [ ]:
import os, json, random, torch
from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
SAMPLE_FRAC = float(os.getenv("SAMPLE_FRAC", "0.05"))

model = CrossEncoder("BAAI/bge-reranker-v2-m3", num_labels=1, max_length=384)
with open("data/cached_schemes.json", encoding="utf-8") as f:
    schemes = json.load(f)

examples = []
for i, s in enumerate(schemes):
    pos = f"{s['name']} ({s['state']}): {s['benefits']}"
    neg_s = schemes[(i + 7) % len(schemes)]
    neg = f"{neg_s['name']} ({neg_s['state']}): {neg_s['benefits']}"
    q = f"How to apply for {s['name']} in {s['state']}?"
    examples.append(InputExample(texts=[q, pos], label=1.0))
    examples.append(InputExample(texts=[q, neg], label=0.0))

random.shuffle(examples)
n_train = max(8, int(len(examples) * SAMPLE_FRAC))
loader = DataLoader(examples[:n_train], shuffle=True, batch_size=8)
model.fit(train_dataloader=loader, epochs=2, warmup_steps=5, output_path="checkpoints/N2_reranker_ft")
print("Saved checkpoint to checkpoints/N2_reranker_ft")


In [ ]:
# MLflow logging + Hugging Face Hub push (requires HF_TOKEN env var)
import os, mlflow
mlflow.set_experiment('yojana-mitra-training')
with mlflow.start_run(run_name='N2_reranker_ft'):
    mlflow.log_params({'base_model': 'BAAI/bge-reranker-v2-m3', 'seed': 42, 'sample_frac': os.getenv('SAMPLE_FRAC', '0.05')})
if os.getenv('HF_TOKEN'):
    from huggingface_hub import HfApi
    print('Ready to upload checkpoints/N2_reranker_ft to yojana-mitra/bge-reranker-v2-m3-welfare-v1')
